# Step 2: Feature Engineering — Demo Notebook
---
**Goal**: Demonstrate trend fitting with LinearRegression, seasonal decomposition with Fourier/DeterministicProcess, and full feature matrix construction.

### Feature Categories:
1. **Trend** — LinearRegression slope per (store, family)
2. **Seasonal** — Fourier harmonics (yearly 6-order, weekly 3-order)
3. **Calendar** — dayofweek, month, year, weekend, etc.
4. **Lag** — sales_lag_1/7/14/28 + promo_lag_1/7
5. **Rolling** — 7d/14d/30d rolling mean, std, min, max
6. **Holiday** — proximity windows around holidays
7. **Oil** — lagged and rolling oil prices
8. **Interactions** — weekend×holiday, promo×weekend

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.deterministic import Fourier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_palette('tab10')
plt.rcParams['figure.dpi'] = 100

# Add code/ to path for config imports
sys.path.insert(0, os.path.abspath(''))
from config import *
from utils import Timer

print('Libraries loaded.')

In [ ]:
# Load cleaned training data
df = pd.read_csv(TRAIN_CLEANED_PATH, parse_dates=['date'])
print(f'Train: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Dates: {df["date"].min().date()} ~ {df["date"].max().date()}')
df.head(3)

## 1. Trend Fitting — LinearRegression per (store, family)

For each (store_nbr, family) combination, fit `sales ~ days_since_start`. Extract slope as a feature.
A positive slope means the product is growing in that store.

In [ ]:
# Demonstrate trend fitting on a single store-family combination
store, family = 44, 'GROCERY I'  # A large store in Quito
sample = df[(df['store_nbr'] == store) & (df['family'] == family)].copy()
sample['days_since_start'] = (sample['date'] - sample['date'].min()).dt.days

lr = LinearRegression()
X = sample['days_since_start'].values.reshape(-1, 1)
y = sample['sales'].values
lr.fit(X, y)
trend_pred = lr.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Full time series with trend
axes[0].scatter(sample['date'], sample['sales'], s=1, alpha=0.3, color='#3498DB')
axes[0].plot(sample['date'], trend_pred, color='#E74C3C', linewidth=2, label=f'Trend (slope={lr.coef_[0]:.3f}/day)')
axes[0].set_title(f'Store {store} — {family}: Sales + Trend', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Sales')
axes[0].legend()

# Detrended (2017 only for clarity)
mask_2017 = sample['date'] >= '2017-01-01'
axes[1].plot(sample.loc[mask_2017, 'date'], sample.loc[mask_2017, 'sales'] - trend_pred[mask_2017],
            linewidth=0.8, color='#2ECC71')
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.5)
axes[1].set_title('Detrended Sales (2017)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Residual (Sales - Trend)')

plt.tight_layout()
plt.show()

print(f'Trend slope: {lr.coef_[0]:.4f} sales/day')
print(f'Trend intercept: {lr.intercept_:.1f}')

## 2. Seasonal Features — Fourier Harmonics

Fourier terms capture cyclical patterns: yearly (period=365.25, 6 harmonics) and weekly (period=7, 3 harmonics).

In [ ]:
# Visualize Fourier yearly harmonics
days = np.arange(0, 365 * 2)  # 2 years
temp_df = pd.DataFrame({'days_idx': days})

dp = Fourier(period=365.25, order=3)
fourier_terms = dp.in_sample(temp_df['days_idx'])

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# First 3 sin/cos pairs
for i in range(3):
    axes[0].plot(days, fourier_terms.iloc[:, 2*i], label=f'sin(order={i+1})', alpha=0.8)
    axes[0].plot(days, fourier_terms.iloc[:, 2*i+1], '--', label=f'cos(order={i+1})', alpha=0.8)
axes[0].set_title('Fourier Harmonics (Yearly, Order 1-3)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Days')
axes[0].legend(loc='upper right', fontsize=8, ncol=2)

# Combined seasonal pattern from 6 harmonics
dp6 = Fourier(period=365.25, order=6)
terms6 = dp6.in_sample(temp_df['days_idx'])
# Sum all Fourier terms to get the seasonal pattern
seasonal_pattern = terms6.sum(axis=1)
axes[1].plot(days, seasonal_pattern, color='#8E44AD', linewidth=1.5)
axes[1].set_title('Combined Seasonal Pattern (6 Harmonics)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Days')
axes[1].set_ylabel('Seasonal Component')

plt.tight_layout()
plt.show()

print(f'Yearly features created: {dp6.in_sample(temp_df.head(1)["days_idx"]).shape[1]} columns (6 sin + 6 cos = 12)')

## 3. Feature Correlation Heatmap

Top 20 features by correlation with sales target.

In [ ]:
# Load the exported feature matrix and check correlations
X_train = pd.read_csv(os.path.join(DATA_FEATURES_DIR, 'X_train.csv'))
y_train = X_train['sales']
X_train.drop(columns=['sales', 'date'], inplace=True, errors='ignore')

# Top 20 features by absolute correlation with sales
corrs = X_train.corrwith(y_train).abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette('Blues_d', len(corrs))
ax.barh(range(len(corrs)), corrs.values[::-1], color=colors)
ax.set_yticks(range(len(corrs)))
ax.set_yticklabels(corrs.index[::-1])
ax.set_title('Top 20 Features by Correlation with Sales', fontsize=13, fontweight='bold')
ax.set_xlabel('|Correlation|')
for i, v in enumerate(corrs.values[::-1]):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 4. Train / Validation Split

Time-based split: training on data before 2017-07-26, validation on 2017-07-26 to 2017-08-15.

In [ ]:
# Visualize the train/validation boundary
daily_total = df.groupby('date')['sales'].sum().reset_index()
daily_total_2017 = daily_total[daily_total['date'] >= '2017-01-01']

fig, ax = plt.subplots(figsize=(14, 4))
split_date = pd.Timestamp('2017-07-26')

train_dates = daily_total_2017[daily_total_2017['date'] < split_date]
valid_dates = daily_total_2017[daily_total_2017['date'] >= split_date]

ax.plot(train_dates['date'], train_dates['sales'] / 1e6, color='#3498DB', linewidth=1, label='Training')
ax.plot(valid_dates['date'], valid_dates['sales'] / 1e6, color='#E74C3C', linewidth=1, label='Validation')
ax.axvline(split_date, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
ax.annotate('Split: 2017-07-26', xy=(split_date, ax.get_ylim()[1]*0.9), fontsize=10, ha='right', color='black')
ax.set_title('Train/Validation Split (2017)', fontsize=13, fontweight='bold')
ax.set_ylabel('Daily Total Sales (Millions)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Training days:   {train_dates.shape[0]}')
print(f'Validation days: {valid_dates.shape[0]}')

---
## Summary

- **163 features** created from 8 categories
- **2.91M training rows**, **37K validation rows**
- **Trend** slopes per (store, family) capture long-term growth
- **Fourier** terms (12 yearly + 6 weekly) capture cyclical seasonality
- **Lag & rolling** features are the most important (top 5 by gain)
- Time-based split ensures no future leakage

→ Proceed to Step 3 for XGBoost training.